# 00. OpenVLA-OFT + LIBERO Reproduction (1순위)

**목표**: 사전학습된 OpenVLA-OFT 체크포인트를 받아 LIBERO 시뮬레이션(MuJoCo)에서 한 태스크 rollout 재현.

**런타임**: Colab → Runtime → Change runtime type → **A100 (권장, 16GB VRAM 필요)**.
T4(16GB)는 빡빡해서 OOM 가능. L4(24GB)도 OK.

## 흐름
1. GPU 확인 + Drive 마운트
2. apt 의존성 (mujoco용 libosmesa, libgl)
3. PyTorch 2.2.0 + openvla-oft + transformers (moojink fork) + LIBERO 설치
4. Sample inference (offline 한 step)
5. LIBERO 한 task rollout → 영상 저장

## 결과 검증
- inference 셀: 7-DOF action chunk 8개 출력
- rollout 셀: success rate + .mp4 영상 (Drive에 저장)

## 1. GPU + Drive

In [ ]:
!nvidia-smi | head -20

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/openvla-pose'
OFT_DIR = f'{PROJECT_ROOT}/openvla-oft'
LIBERO_DIR = f'{PROJECT_ROOT}/LIBERO'
HF_CACHE = f'{PROJECT_ROOT}/hf_cache'
VIDEO_DIR = f'{PROJECT_ROOT}/rollout_videos'
for p in [PROJECT_ROOT, HF_CACHE, VIDEO_DIR]:
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
# MuJoCo headless rendering
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
print('Drive ready. PROJECT_ROOT =', PROJECT_ROOT)

## 2. apt 의존성 (MuJoCo headless)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev patchelf
!echo 'apt OK'

## 3. PyTorch 2.2.0 + openvla-oft

Colab 기본 Python (3.11)에서도 대부분 동작합니다. Python 3.10 강제는 conda 필요한데 Colab에선 까다로움.

In [ ]:
!pip install -q torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Clone openvla-oft into Drive (영구 보관 → 다음 세션에 git pull만)
if not os.path.exists(OFT_DIR):
    !git clone https://github.com/moojink/openvla-oft.git $OFT_DIR
else:
    print('openvla-oft already cloned at', OFT_DIR)
%cd $OFT_DIR
!git log -1 --oneline

In [ ]:
# pip install openvla-oft (editable)
%cd $OFT_DIR
!pip install -q -e .
# transformers의 moojink fork (OpenVLA-OFT 호환)
!pip install -q git+https://github.com/moojink/transformers-openvla-oft.git
# tokenizers, peft 호환 버전
!pip install -q tokenizers==0.19.1 peft==0.11.1 accelerate==0.27.2 sentencepiece einops draccus==0.8.0

## 4. LIBERO 설치

In [ ]:
if not os.path.exists(LIBERO_DIR):
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git $LIBERO_DIR
%cd $LIBERO_DIR
!pip install -q -e .
%cd $OFT_DIR
# OFT가 LIBERO 평가에 추가로 필요한 패키지
!pip install -q -r experiments/robot/libero/libero_requirements.txt
print('LIBERO installed')

## 5. flash-attn (A100/H100 only — 실패해도 OFT는 eager로 동작)

In [ ]:
import torch
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -q packaging ninja
    !pip install -q 'flash-attn==2.5.5' --no-build-isolation
else:
    print('SM<8 GPU — flash-attn 건너뜀 (eager attention 사용)')

## 6. Sample Inference (offline)

실제 시뮬을 돌리기 전에, 사전학습 체크포인트가 정상 로드되고 action을 출력하는지 확인.
샘플 observation .pkl이 repo에 포함됨.

In [ ]:
%cd $OFT_DIR
import sys
if OFT_DIR not in sys.path: sys.path.insert(0, OFT_DIR)

import pickle
from experiments.robot.libero.run_libero_eval import GenerateConfig
from experiments.robot.openvla_utils import (
    get_action_head, get_processor, get_proprio_projector, get_vla, get_vla_action,
)
from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM

cfg = GenerateConfig(
    pretrained_checkpoint='moojink/openvla-7b-oft-finetuned-libero-spatial',
    use_l1_regression=True,
    use_diffusion=False,
    use_film=False,
    num_images_in_input=2,
    use_proprio=True,
    load_in_8bit=False,
    load_in_4bit=False,
    center_crop=True,
    num_open_loop_steps=NUM_ACTIONS_CHUNK,
    unnorm_key='libero_spatial_no_noops',
)

print('Loading VLA...')
vla = get_vla(cfg)
processor = get_processor(cfg)
action_head = get_action_head(cfg, llm_dim=vla.llm_dim)
proprio_projector = get_proprio_projector(cfg, llm_dim=vla.llm_dim, proprio_dim=PROPRIO_DIM)

with open('experiments/robot/libero/sample_libero_spatial_observation.pkl', 'rb') as f:
    observation = pickle.load(f)

actions = get_vla_action(cfg, vla, processor, observation, observation['task_description'],
                          action_head, proprio_projector)
print(f'\nGenerated action chunk ({len(actions)} actions, 7-DOF each):')
for i, a in enumerate(actions):
    print(f'  step {i}: {a}')

## 7. **LIBERO Rollout (1 task)** — MuJoCo 시뮬에서 정책 평가

기본 500 trial 대신 빠른 검증을 위해 5 trial만. 약 5-15분 소요.

In [ ]:
%cd $OFT_DIR
# 영상 저장 디렉토리
os.environ['LIBERO_VIDEO_DIR'] = VIDEO_DIR

# 빠른 sanity rollout: 1 task x 5 trials
!python experiments/robot/libero/run_libero_eval.py \
  --pretrained_checkpoint moojink/openvla-7b-oft-finetuned-libero-spatial \
  --task_suite_name libero_spatial \
  --num_trials_per_task 5 \
  --use_l1_regression True \
  --num_images_in_input 2 \
  --use_proprio True \
  --center_crop True

## 8. 영상 확인 (Colab inline)

In [ ]:
import glob
from IPython.display import Video, display

# OFT eval script가 저장하는 위치 (보통 ./rollouts/ 또는 wandb)
for d in ['./rollouts', '/content/rollouts', VIDEO_DIR]:
    vids = sorted(glob.glob(f'{d}/**/*.mp4', recursive=True))
    if vids:
        print(f'Found {len(vids)} videos in {d}')
        # Drive로 복사
        for v in vids[:3]:
            !cp '$v' '$VIDEO_DIR/'
            print('copied:', v)
        # 첫 영상 표시
        display(Video(vids[0], embed=True, width=480))
        break
else:
    print('영상 못 찾음. ls -la ./rollouts/')
    !ls -la ./rollouts/ 2>&1 | head -20